# 📊 Análise de Sentimento - Unidade C

## Objetivo
Analisar a percepção dos pacientes a partir das avaliações públicas.

## Metodologia
- NLP (análise de sentimento)
- análise temporal
- análise de frequência de palavras

In [11]:
import pandas as pd
import numpy as np

In [13]:
from transformers import pipeline

In [14]:
import re
from collections import Counter

In [15]:
df_altiplano = pd.read_csv("01_Altiplano_Google_Reviews.csv")
df_altiplano.head()

,unidade,autor,rating,data,comentario
0,Altiplano,Kcris Brilhante,5,2026,"Gente, só queria deixar registrado o meu MUITO..."
1,Altiplano,Maria Gizelia Macedo,5,2026,Ótimo atendimento e muita presteza
2,Altiplano,Ana Paula Rodrigues,5,2026,Avaliação sem texto
3,Altiplano,Alcione ferreira De araujo,5,2026,Avaliação sem texto
4,Altiplano,Alba,5,2026,Avaliação sem texto


In [16]:
df_altiplano.info()
df_altiplano.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   unidade     309 non-null    object
 1   autor       309 non-null    object
 2   rating      309 non-null    int64 
 3   data        309 non-null    int64 
 4   comentario  309 non-null    object
dtypes: int64(2), object(3)
memory usage: 12.2+ KB


(309, 5)

In [17]:
df_altiplano["rating"].value_counts().sort_index()

,count
rating,
1,37
2,3
3,7
4,19
5,243


In [18]:
df_altiplano["rating"].value_counts(normalize=True).sort_index() * 100

,proportion
rating,
1,11.974110
2,0.970874
3,2.265372
4,6.148867
5,78.640777


In [19]:
df_altiplano["data"].value_counts().sort_index()

,count
data,
2020,3
2021,14
2022,44
2023,21
2024,11
2025,63
2026,153


In [20]:
df_altiplano.groupby("data")["rating"].mean().round(2)

,rating
data,
2020,5.00
2021,2.43
2022,4.02
2023,3.67
2024,4.00
2025,4.16
2026,4.88


In [21]:
df_altiplano["comentario"] = df_altiplano["comentario"].fillna("Avaliação sem texto")

df_sent_alt = df_altiplano[df_altiplano["comentario"] != "Avaliação sem texto"].copy()

df_crit_alt = df_sent_alt[df_sent_alt["data"].isin([2023, 2024, 2025, 2026])].copy()

len(df_crit_alt)

90

In [23]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [24]:
def analisar_sentimento(texto):
    resultado = sentiment_pipeline(
        str(texto),
        truncation=True,
        max_length=128
    )[0]
    return resultado["label"]

In [25]:
df_crit_alt["sentimento"] = df_crit_alt["comentario"].apply(analisar_sentimento)

df_crit_alt["sentimento"].value_counts(normalize=True) * 100

,proportion
sentimento,
POS,63.333333
NEG,27.777778
NEU,8.888889


In [28]:
import re
from collections import Counter

stopwords = [
    "de","da","do","das","dos","a","o","as","os","e","é","em","para",
    "com","que","não","na","no","uma","um","mais","foi","por","se",
    "ao","à","às","como","já","eu","ele","ela","me","minha","meu",
    "muito","porque","isso","está","estava","são","ser","tem","pra",
    "pelo","pela","até","fui","era"
]

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stopwords and len(p) > 2]
    return palavras

In [29]:
df_pos_alt = df_crit_alt[df_crit_alt["sentimento"] == "POS"].copy()

todas_palavras_pos = []

for comentario in df_pos_alt["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras_pos.extend(palavras)

Counter(todas_palavras_pos).most_common(20)

[('atendimento', 36),
 ('excelente', 17),
 ('ótimo', 13),
 ('dra', 7),
 ('clínica', 6),
 ('recepção', 6),
 ('médica', 6),
 ('bem', 6),
 ('médicos', 6),
 ('atendida', 5),
 ('organizado', 5),
 ('profissional', 5),
 ('equipe', 4),
 ('bom', 4),
 ('rápido', 4),
 ('competente', 4),
 ('atencioso', 4),
 ('profissionais', 3),
 ('atenciosos', 3),
 ('recomendo', 3)]

In [30]:
df_neg_alt = df_crit_alt[df_crit_alt["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg_alt["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 16),
 ('médico', 9),
 ('clínica', 7),
 ('atende', 6),
 ('vez', 6),
 ('mas', 6),
 ('unimed', 6),
 ('recepção', 5),
 ('sei', 5),
 ('unidade', 5),
 ('sem', 5),
 ('vezes', 4),
 ('atender', 4),
 ('nem', 4),
 ('fazer', 4),
 ('atendentes', 4),
 ('falta', 4),
 ('telefone', 4),
 ('aqui', 3),
 ('espera', 3)]